# 03 - Feature Engineering
Input: `data/clean.csv`. Output: `data/features.csv`.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "train_v9rqX0R.csv").exists():
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / "data" / "clean.csv")
print(df.shape)

(14204, 13)


## Item features
- `Item_Cat`: first 2 chars of the ID (FD food, DR drink, NC non-consumable)
- Non-consumables can't have a fat content -> `Non-Edible`
- `Item_Count`: how many outlets stock the item (popularity proxy)

In [2]:
df["Item_Cat"] = df["Item_Identifier"].str[:2]
df.loc[df["Item_Cat"] == "NC", "Item_Fat_Content"] = "Non-Edible"
df["Item_Count"] = df.groupby("Item_Identifier")["Item_Identifier"].transform("count")
print(df.groupby(["Item_Cat", "Item_Fat_Content"]).size())

Item_Cat  Item_Fat_Content
DR        Low Fat             1208
          Regular              109
FD        Low Fat             5291
          Regular             4910
NC        Non-Edible          2686
dtype: int64


## Outlet features
- `Outlet_Age`: data is from 2013
- `Vis_Ratio`: item visibility relative to the outlet's average (shelf share is only
  meaningful within a store)

In [3]:
df["Outlet_Age"] = 2013 - df["Outlet_Establishment_Year"]
df["Vis_Ratio"] = df["Item_Visibility"] / df.groupby("Outlet_Identifier")["Item_Visibility"].transform("mean")

## Price features
- `MRP_per_wt`: price per unit weight
- `MRP_vs_type`: price relative to the average of its item type

In [4]:
df["MRP_per_wt"] = df["Item_MRP"] / df["Item_Weight"]
df["MRP_vs_type"] = df["Item_MRP"] / df.groupby("Item_Type")["Item_MRP"].transform("mean")

## Encode categoricals (integer codes; model treats them as categorical)

In [5]:
CATS = ["Item_Fat_Content", "Item_Type", "Outlet_Identifier", "Outlet_Size",
        "Outlet_Location_Type", "Outlet_Type", "Item_Cat"]
keep_ids = df[["Item_Identifier", "Outlet_Identifier"]].rename(
    columns={"Outlet_Identifier": "Outlet_ID_raw"})
for c in CATS:
    df[c] = df[c].astype("category").cat.codes
df = df.join(keep_ids["Outlet_ID_raw"])

## Save

In [6]:
df = df.drop(columns="Outlet_Establishment_Year")   # replaced by Outlet_Age
df.to_csv(ROOT / "data" / "features.csv", index=False)
print("saved features.csv", df.shape)
df.head()

saved features.csv (14204, 19)


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,src,Item_Cat,Item_Count,Outlet_Age,Vis_Ratio,MRP_per_wt,MRP_vs_type,Outlet_ID_raw
0,FDA15,9.30,0,0.016047,4,249.8092,9,1,0,1,3735.1380,train,1,9,14,0.246785,26.861204,1.696716,OUT049
1,DRC01,5.92,2,0.019278,14,48.2692,3,1,2,2,443.4228,train,0,9,4,0.296089,8.153581,0.356484,OUT018
2,FDN15,17.50,0,0.016760,10,141.6180,9,1,0,1,2097.2700,train,1,10,14,0.257747,8.092457,1.009458,OUT049
3,FDX07,19.20,2,0.022930,6,182.0950,0,2,2,0,732.3800,train,1,9,15,0.216666,9.484115,1.261883,OUT010
4,NCD19,8.93,1,0.014670,9,53.8614,1,0,2,1,994.7052,train,2,9,26,0.226324,6.031512,0.362126,OUT013
